# Day 7: Incremental Processing & CDC Upsert Engine

This notebook demonstrates idempotent UPSERT/DELETE logic using Iceberg's `MERGE INTO` SQL commands.

In [ ]:
!pip install pyspark pyiceberg
import os
os.environ['SPARK_VERSION'] = '3.5'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit

spark = SparkSession.builder \
    .appName("Day07_Incremental_Upsert") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/iceberg_warehouse") \
    .getOrCreate()

print("Spark session created.")

In [ ]:
# Create target Iceberg table
spark.sql("CREATE DATABASE IF NOT EXISTS local.ehdip_silver_db")

spark.sql("""
CREATE TABLE IF NOT EXISTS local.ehdip_silver_db.condition_occurrence (
    condition_occurrence_id STRING,
    person_id STRING,
    condition_concept_id INT,
    updated_at TIMESTAMP,
    cdc_op STRING
)
USING iceberg
""")

# Insert initial state
spark.sql("""
INSERT INTO local.ehdip_silver_db.condition_occurrence VALUES 
('C001', 'P123', 32020, TIMESTAMP '2023-01-01 10:00:00', 'c'),
('C002', 'P124', 32020, TIMESTAMP '2023-01-01 10:00:00', 'c')
""")

print("--- INITIAL STATE ---")
spark.sql("SELECT * FROM local.ehdip_silver_db.condition_occurrence").show()

# Create incoming updates batch
updates_data = [
    ('C001', 'P123', 32021, '2023-01-02 10:00:00', 'u'), # Update C001
    ('C002', 'P124', 32020, '2023-01-02 10:00:00', 'd'), # Delete C002
    ('C003', 'P125', 32020, '2023-01-02 10:00:00', 'c')  # Insert C003
]
updates_df = spark.createDataFrame(updates_data, ["condition_occurrence_id", "person_id", "condition_concept_id", "updated_at", "cdc_op"])
updates_df = updates_df.withColumn("updated_at", col("updated_at").cast("timestamp"))
updates_df.createOrReplaceTempView("updates")

print("--- INCOMING UPDATES ---")
updates_df.show()

# Perform MERGE INTO
merge_query = """
MERGE INTO local.ehdip_silver_db.condition_occurrence t
USING updates s
ON t.condition_occurrence_id = s.condition_occurrence_id
WHEN MATCHED AND s.cdc_op = 'd' AND s.updated_at > t.updated_at THEN
    DELETE
WHEN MATCHED AND (s.cdc_op = 'u' OR s.cdc_op IS NULL) AND s.updated_at > t.updated_at THEN
    UPDATE SET *
WHEN NOT MATCHED AND (s.cdc_op = 'c' OR s.cdc_op = 'u' OR s.cdc_op IS NULL) THEN
    INSERT *
"""
spark.sql(merge_query)

print("--- FINAL STATE AFTER UPSERT/DELETE ---")
spark.sql("SELECT * FROM local.ehdip_silver_db.condition_occurrence").show()
